# CrowdGuard — Version 2 Complete Kaggle Run

This notebook contains the complete setup, environment checks, v2 smoke tests,
pipeline smoke test, validation tests, and the 20-client CrowdGuard v2 experiment.

**This notebook is designed for Kaggle "Save & Run All": it runs top to bottom and stops
at the first failing step (tests, smoke tests, experiment, validation).
Set `PMR` in the parameter cell before saving each version.**

This is the v2 experiment. The completed v1 experiment is documented separately
in `RESULTS_V1.md`.

## Main v2 changes

- Frequency-based final-M selection
- Default `final_M_tau = 0.5`
- Two-phase commit-reveal for validator votes
- 2,500-image KD/reference split
- 7,500-image held-out evaluation split
- Held-out-only ASR evaluation
- Per-round crash-insurance checkpoints
- Reproducibility metadata
- Source-file SHA-256 hashes
- Handoff validation
- Handoff evaluation

## 0. Kaggle requirements

- GPU enabled
- Internet enabled
- Python 3.10 environment created with `uv`
- OpenFL 1.9
- Metaflow 2.7.15
- PyTorch with CUDA

The v1 run established that the default Kaggle Python environment should not
be used directly because Metaflow 2.7.15 requires the Python 3.10 environment
used by this project.

In [1]:
# Basic Kaggle/GPU check

import sys
import os

print("Notebook Python:", sys.version)

!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!free -h
!df -h /kaggle/working

Notebook Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
/bin/bash: line 1: nvidia-smi: command not found
               total        used        free      shared  buff/cache   available
Mem:            31Gi       1.2Gi        25Gi       1.2Mi       5.0Gi        30Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G   64K   20G   1% /kaggle/working


In [2]:
# Make sure uv is available

!command -v uv || python -m pip install -q uv
!uv --version

/usr/local/bin/uv
uv 0.12.15 (x86_64-unknown-linux-gnu)


In [3]:
# Clone the exact CrowdGuard v2 branch

# After merging the Dirichlet PR and fixing `config.partition`, paste the FULL hash of the
# fix commit here so every run in the matrix uses identical code.  Leave "" to use the branch head.
PIN_COMMIT = ""

!rm -rf /kaggle/working/Verifiable-Federated-Unlearning

!git clone \
    -b crowdguard-branch \
    https://github.com/clearlynew/Verifiable-Federated-Unlearning.git \
    /kaggle/working/Verifiable-Federated-Unlearning

if PIN_COMMIT:
    !git -C /kaggle/working/Verifiable-Federated-Unlearning checkout $PIN_COMMIT

!git -C /kaggle/working/Verifiable-Federated-Unlearning branch --show-current
!git -C /kaggle/working/Verifiable-Federated-Unlearning log -1 --oneline

Cloning into '/kaggle/working/Verifiable-Federated-Unlearning'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 104 (delta 42), reused 70 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 7.42 MiB | 35.67 MiB/s, done.
Resolving deltas: 100% (42/42), done.
crowdguard-branch
4e2a879 (HEAD -> crowdguard-branch, origin/crowdguard-branch) Added name error fix and two more if cases in validation files


In [4]:
# Enter the CrowdGuard project

%cd /kaggle/working/Verifiable-Federated-Unlearning/CrowdGuard

!pwd
!ls -lah

/kaggle/working/Verifiable-Federated-Unlearning/CrowdGuard
/kaggle/working/Verifiable-Federated-Unlearning/CrowdGuard
total 412K
drwxr-xr-x 8 root root 4.0K Oct  6 07:23 .
drwxr-xr-x 6 root root 4.0K Oct  6 07:23 ..
-rw-r--r-- 1 root root 5.8K Oct  6 07:23 adaptive_backdoor.py
drwxr-xr-x 2 root root 4.0K Oct  6 07:23 Amsterdam
drwxr-xr-x 2 root root 4.0K Oct  6 07:23 Bangalore
drwxr-xr-x 2 root root 4.0K Oct  6 07:23 Chandler
-rw-r--r-- 1 root root  32K Oct  6 07:23 cifar10_crowdguard.py
-rw-r--r-- 1 root root 5.0K Oct  6 07:23 commit_reveal.py
-rw-r--r-- 1 root root  22K Oct  6 07:23 CrowdGuardClientValidation.py
drwxr-xr-x 2 root root 4.0K Oct  6 07:23 Detroit
drwxr-xr-x 2 root root 4.0K Oct  6 07:23 director
-rw-r--r-- 1 root root 6.1K Oct  6 07:23 evaluate_handoff.py
-rw-r--r-- 1 root root 4.2K Oct  6 07:23 experiment_config.py
-rw-r--r-- 1 root root  747 Oct  6 07:23 ExperimentMatrix.md
-rw-r--r-- 1 root root  55K Oct  6 07:23 FederatedCrowdGuard.ipynb
-rw-r--r-- 1 root root   41 

## 1. Create the Python 3.10 environment

The v2 run uses Python 3.10.21.

The environment is kept separate from Kaggle's notebook Python.

In [5]:
# Install/find Python 3.10 through uv

!uv python install 3.10

!rm -rf /kaggle/working/crowdguard-env

!uv venv \
    /kaggle/working/crowdguard-env \
    --python 3.10

!/kaggle/working/crowdguard-env/bin/python --version

Installed Python 3.10.21 in 1.71s
 + cpython-3.10.21-linux-x86_64-gnu (python3.10)
Using CPython 3.10.21
Creating virtual environment at: /kaggle/working/crowdguard-env
Activate with: source /kaggle/working/crowdguard-env/bin/activate
Python 3.10.21


In [6]:
# Install the core versions required by the project

!uv pip install \
    --python /kaggle/working/crowdguard-env/bin/python \
    "openfl==1.9" \
    "metaflow==2.7.15" \
    "setuptools<81"

Using Python 3.10.21 environment at: /kaggle/working/crowdguard-env
Resolved 56 packages in 684ms                                        
Prepared 56 packages in 1.50s                                            
░░░░░░░░░░░░░░░░░░░░ [0/56] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 56 packages in 826ms                              
 + astroid==4.3.3
 + blinker==1.9.0
 + boto3==1.43.108
 + botocore==1.43.108
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cloudpickle==3.1.2
 + cryptography==50.0.2
 + dill==0.4.1
 + dynaconf==3.3.5
 + flask==3.1.1
 + flatten-json==0.1.14
 + grpcio==1.65.5
 + idna==3.20
 + isort==

In [7]:
# Install the remaining project dependencies

!uv pip install \
    --python /kaggle/working/crowdguard-env/bin/python \
    torch \
    torchvision \
    numpy \
    scipy \
    scikit-learn \
    pandas \
    matplotlib \
    ray \
    dill \
    tabulate \
    nbformat \
    nbdev

Using Python 3.10.21 environment at: /kaggle/working/crowdguard-env
Resolved 104 packages in 466ms                                       
Prepared 76 packages in 44.80s                                           
░░░░░░░░░░░░░░░░░░░░ [0/76] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 76 packages in 56.56s                             
 + anyio==4.15.1
 + asttokens==3.0.2
 + astunparse==1.6.3
 + attrs==26.1.0
 + build==1.6.1
 + contourpy==1.3.2
 + cuda-bindings==13.4.3
 + cuda-pathfinder==1.8.3
 + cuda-toolkit==13.0.3.0
 + cycler==0.12.1
 + decorator==5.3.1
 + exceptiongroup==1.3.1
 + execnb==0.3.3
 + executing==2.2.1
 + fastcore==2.2.33
 + fastgit==0

In [8]:
# Pin the scikit-learn version used by the reproducible environment

!uv pip install \
    --python /kaggle/working/crowdguard-env/bin/python \
    --force-reinstall \
    "scikit-learn==1.6.1"

Using Python 3.10.21 environment at: /kaggle/working/crowdguard-env
Resolved 6 packages in 154ms                                         
Prepared 6 packages in 231ms                                             
Uninstalled 6 packages in 129ms
░░░░░░░░░░░░░░░░░░░░ [0/6] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 6 packages in 1.23s                               
 ~ cloudpickle==3.1.2
 ~ joblib==1.6.0
 ~ numpy==2.2.6
 - scikit-learn==1.7.2
 + scikit-learn==1.6.1
 ~ scipy==1.15.3
 ~ threadpoolctl==3.7.0


In [9]:
# Metaflow needs a user identity in the Kaggle environment

import os

os.environ["USERNAME"] = "kaggle"
os.environ["USER"] = "kaggle"

print("USERNAME =", os.environ["USERNAME"])
print("USER =", os.environ["USER"])

USERNAME = kaggle
USER = kaggle


### Helper: run a script and stop the notebook if it fails

A failing `!command` does NOT stop a notebook, so with Save & Run All a failed
test would let the 4-hour experiment start anyway, and a crashed experiment
would still produce a "successful" version. `run_checked` streams the output
live (and optionally to a log file) and raises if the script exits non-zero.

In [10]:
import subprocess

python = "/kaggle/working/crowdguard-env/bin/python"


def run_checked(*args, log=None):
    """Run `python -u <args>` in the project environment; raise on failure."""
    proc = subprocess.Popen(
        [python, "-u", *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    log_file = open(log, "w", encoding="utf-8") if log else None
    try:
        for line in proc.stdout:
            print(line, end="")
            if log_file:
                log_file.write(line)
                log_file.flush()
    finally:
        if log_file:
            log_file.close()
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"{' '.join(args[:2])} failed with exit code {code}")

In [11]:
# Verify the exact environment and CUDA

check = r'''
import sys
import torch
import openfl
import metaflow
import numpy
import scipy
import sklearn
import pandas

print("Python:", sys.version.split()[0])
print("OpenFL:", openfl.__version__)
print("Metaflow:", metaflow.__version__)
print("PyTorch:", torch.__version__)
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("pandas:", pandas.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
'''

run_checked("-c", check)

!nvidia-smi

/kaggle/working/crowdguard-env/lib/python3.10/site-packages/metaflow/metaflow_config.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Python: 3.10.21
OpenFL: 1.9
Metaflow: 2.7.15
PyTorch: 2.14.1+cu130
NumPy: 2.2.6
SciPy: 1.15.3
sklearn: 1.6.1
pandas: 2.3.3
CUDA available: False
/bin/bash: line 1: nvidia-smi: command not found


Expected important versions:

```text
Python: 3.10.21
OpenFL: 1.9
Metaflow: 2.7.15
PyTorch: 2.14.1+cu130
NumPy: 2.2.6
SciPy: 1.15.3
sklearn: 1.6.1
pandas: 2.3.3
```

## 2. Verify the v2 implementation

The following files should exist before running the experiment:

- `cifar10_crowdguard.py`
- `CrowdGuardClientValidation.py`
- `adaptive_backdoor.py`
- `lightweight_resnet18.py`
- `experiment_config.py`
- `commit_reveal.py`
- `handoff_utils.py`
- `validate_handoff.py`
- `evaluate_handoff.py`
- `test_handoff_utils.py`
- `test_two_phase_flow.py`
- `test_validate_handoff.py`
- `local_smoke_test.py`
- `local_pipeline_smoke.py`

In [12]:
from pathlib import Path

required_files = [
    "cifar10_crowdguard.py",
    "CrowdGuardClientValidation.py",
    "adaptive_backdoor.py",
    "lightweight_resnet18.py",
    "experiment_config.py",
    "commit_reveal.py",
    "handoff_utils.py",
    "validate_handoff.py",
    "evaluate_handoff.py",
    "test_handoff_utils.py",
    "test_two_phase_flow.py",
    "test_validate_handoff.py",
    "local_smoke_test.py",
    "local_pipeline_smoke.py",
]

missing = [
    filename
    for filename in required_files
    if not Path(filename).exists()
]

if missing:
    print("Missing files:")
    for filename in missing:
        print(" -", filename)
    raise FileNotFoundError("Required v2 files are missing.")

print("All required v2 files are present.")

All required v2 files are present.


## 3. Run v2 validation tests

These tests should pass before launching the expensive 20-client experiment.

In [13]:
run_checked("test_handoff_utils.py")

test_changed_vote_is_rejected (__main__.CommitRevealTests) ... ok
test_commitment_bound_to_validator (__main__.CommitRevealTests) ... ok
test_honest_round_all_valid (__main__.CommitRevealTests) ... ok
test_missing_and_malformed (__main__.CommitRevealTests) ... ok
test_replay_in_other_round_is_rejected (__main__.CommitRevealTests) ... ok
test_wrong_context_is_rejected (__main__.CommitRevealTests) ... ok
test_report (__main__.DetectionReportTests) ... ok
test_digest_properties (__main__.DigestTests) ... ok
test_bad_arguments (__main__.FinalMTests) ... ok
test_counts (__main__.FinalMTests) ... ok
test_float_edge (__main__.FinalMTests) ... ok
test_frequency_boundaries (__main__.FinalMTests) ... ok
test_frequency_drops_one_off_flags (__main__.FinalMTests) ... ok
test_union_keeps_one_off_flags (__main__.FinalMTests) ... ok
test_bad_size (__main__.SplitTests) ... ok
test_deterministic_and_seed_sensitive (__main__.SplitTests) ... ok
test_sizes_disjoint_stratified (__main__.SplitTests) ... ok
t

In [14]:
run_checked("test_validate_handoff.py")

test_changed_revealed_vote_is_caught (__main__.ValidatorTests) ... ok
test_clean_handoff_passes (__main__.ValidatorTests) ... ok
test_digest_mismatch_is_caught (__main__.ValidatorTests) ... ok
test_forged_detection_is_caught (__main__.ValidatorTests) ... ok
test_missing_client_is_caught (__main__.ValidatorTests) ... ok
test_old_schema_is_refused (__main__.ValidatorTests) ... ok
test_overlapping_partitions_are_caught (__main__.ValidatorTests) ... ok
test_overlapping_test_split_is_caught (__main__.ValidatorTests) ... ok
test_rejected_reveal_is_reported_when_consistent (__main__.ValidatorTests) ... ok
test_swapped_commitment_list_is_caught (__main__.ValidatorTests) ... ok
test_tamper_flag_warns (__main__.ValidatorTests) ... ok
test_theta_k_mismatch_is_caught (__main__.ValidatorTests) ... ok
test_unconfirmed_commitment_is_reported (__main__.ValidatorTests) ... ok
test_weight_identity_break_is_caught (__main__.ValidatorTests) ... ok
test_wrong_counts_are_caught (__main__.ValidatorTests) ...

In [15]:
run_checked("test_two_phase_flow.py")

/kaggle/working/crowdguard-env/lib/python3.10/site-packages/metaflow/metaflow_config.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Aggregator step "start" registered
Collaborator step "local_validation" registered
Aggregator step "collect_commitments" registered
Collaborator step "reveal_votes" registered
Aggregator step "defend" registered
Aggregator step "end" registered
test_validator_refuses_to_reveal_if_commitment_was_altered (__main__.SubstitutedCommitmentTests) ... Creating local datastore in current directory (/tmp/tmp5ct82w35/.metaflow)

Calling start

Calling local_validation
Should transfer from local_validation to collect_commitments

Calling local_validation
Should transfer from local_validation to collect_commitments

Calling local_validation
Should t

## 4. Smoke test

This checks:

- lightweight model
- trigger generation
- benign local update
- adaptive malicious local update
- CrowdGuard HLBIM/pruning path
- historical update capture
- detection-only handoff

This is only a sanity check and is **not** the experimental result.

In [16]:
run_checked("local_smoke_test.py")

[1/6] Model parameters: 275,730
[2/6] Forward + internal states: OK (10 states)
[3/6] Random trigger: OK at (18, 1), target=6
[4/6] Benign + adaptive malicious local updates: OK
Distance: cosine, use y 2: [np.float64(3.7657714279794527), np.float64(0.5257937791406198)]
Distance: cosine, use x 2: [np.float64(19.100749643515236), np.float64(0.5257937791406202)]
/kaggle/working/crowdguard-env/lib/python3.10/site-packages/scipy/stats/_morestats.py:3057: RuntimeWarning: divide by zero encountered in scalar divide
  W = numer / denom
Distance: cosine, use y 1: [np.float64(12.24744871391591)]
Distance: cosine, use x 2: [np.float64(12.247448713915906), np.float64(0.0)]
/kaggle/working/crowdguard-env/lib/python3.10/site-packages/scipy/stats/_morestats.py:3057: RuntimeWarning: divide by zero encountered in scalar divide
  W = numer / denom
Distance: euclid, use y 2: [np.float64(9.231046695191218), np.float64(0.2514376430033056)]
Distance: euclid, use x 2: [np.float64(14.821162881930277), np.floa

## 5. Local pipeline smoke test

This exercises the workflow-level pipeline before the expensive full experiment.

In [17]:
run_checked("local_pipeline_smoke.py")

Distance: cosine, use y 2: [np.float64(11.230569430856624), np.float64(0.7216350454400204)]
Distance: cosine, use x 2: [np.float64(0.7216350454400204), np.float64(10.15451747259592)]
/kaggle/working/crowdguard-env/lib/python3.10/site-packages/scipy/stats/_morestats.py:3057: RuntimeWarning: divide by zero encountered in scalar divide
  W = numer / denom
Distance: cosine, use y 1: [np.float64(12.247448713915906)]
Distance: cosine, use x 2: [np.float64(12.24744871391591), np.float64(0.0)]
/kaggle/working/crowdguard-env/lib/python3.10/site-packages/scipy/stats/_morestats.py:3057: RuntimeWarning: divide by zero encountered in scalar divide
  W = numer / denom
Distance: euclid, use y 2: [np.float64(16.2162759545069), np.float64(3.205347500825307)]
Distance: euclid, use x 2: [np.float64(3.205347500825307), np.float64(3.6542917908729002)]
Distance: euclid, use y 1: [np.float64(12.24744871391591)]
Distance: euclid, use x 2: [np.float64(12.24744871391591), np.float64(0.0)]
/kaggle/working/crowdg

## 6. Inspect the v2 configuration

The intended primary v2 experiment is:

| Parameter | Value |
|---|---:|
| Clients | 20 |
| Training samples/client | 2,500 |
| FL rounds | 10 |
| Local epochs | 10 |
| Batch size | 64 |
| Optimizer | SGD |
| Learning rate | 0.01 |
| Momentum | 0.9 |
| PMR | 0.05 |
| Poison rate | 0.10 |
| Attack alpha | 0.70 |
| Trigger | 6 × 6 |
| Seed | 10 |
| Final-M rule | frequency |
| Final-M tau | 0.5 |

The v2 implementation uses detection-only aggregation: detected clients are
still included in FedAvg so that the resulting global model remains suitable
for the downstream unlearning workflow.

## 7. Resource check before the full run

The v1 experiment took approximately 4 hours 14 minutes.

The v2 experiment is similarly expected to be a long-running experiment.

Do not close the Kaggle session once the full experiment has started.

In [19]:
!nvidia-smi
!free -h
!df -h /kaggle/working

/bin/bash: line 1: nvidia-smi: command not found
               total        used        free      shared  buff/cache   available
Mem:            31Gi       1.4Gi       5.3Gi       1.2Mi        25Gi        29Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G  6.0G   14G  31% /kaggle/working


# 8. START THE FULL V2 EXPERIMENT

This is the long-running cell.

If you run this notebook interactively, run cells one at a time; do not start this
cell twice.  With Save & Run All it simply runs in sequence.

The console output is also saved to:

`outputs/v2_full_run.log`

The v2 implementation saves completed-round artifacts under:

`outputs/rounds/`

These provide crash insurance if the run fails after one or more completed rounds.

### Experiment matrix: one row = one saved version

Set the values in the next cell, then **Save & Run All**. Use one copy of this notebook per row
(Copy & Edit), and do not edit a notebook while one of its versions is still queued.

| Row | Data | Attacker | `PMR` | `SEED` | `PARTITION` | `DIRICHLET_ALPHA` | Status |
|---|---|---|---:|---:|---|---:|---|
| 1 | IID | No (clean baseline) | 0 | 10 | iid | - | todo |
| 2 | IID | Yes | 0.05 | 10 | iid | - | done |
| 3 | IID | Yes | 0.10 | 10 | iid | - | done |
| 4 | IID | Yes | 0.45 | 10 | iid | - | done |
| 5 | IID | No | 0 | 42 | iid | - | todo |
| 6 | IID | Yes | 0.10 | 42 | iid | - | todo |
| 7 | Dirichlet | No | 0 | 10 | dirichlet | 0.5 | todo |
| 8 | Dirichlet | Yes | 0.10 | 10 | dirichlet | 0.5 | todo |
| 9 | Dirichlet | Yes | 0.45 | 10 | dirichlet | 0.5 | todo |
| 10 | Dirichlet | No | 0 | 10 | dirichlet | 0.1 | todo |
| 11 | Dirichlet | Yes | 0.10 | 10 | dirichlet | 0.1 | todo |
| 12 | Dirichlet | Yes | 0.45 | 10 | dirichlet | 0.1 | todo |

`PMR = 0` means no malicious client at all. In a clean run every flagged client is a false positive.

In [20]:
# ===== Run parameters: set these for the matrix row you are running =====
PMR = 0               # 0 = clean baseline (no attacker); 0.05 / 0.10 / 0.45 = attack
SEED = 10                # 10 or 42
PARTITION = "iid"        # "iid" or "dirichlet"
#DIRICHLET_ALPHA = 0.5    # only used when PARTITION == "dirichlet": 0.5 or 0.1

assert PARTITION in ("iid", "dirichlet"), PARTITION
assert 0 <= PMR < 0.5, PMR
RUN_NAME = (f"pmr{PMR}_seed{SEED}_{PARTITION}"
            + (f"_alpha{DIRICHLET_ALPHA}" if PARTITION == "dirichlet" else ""))
print("RUN_NAME:", RUN_NAME)

RUN_NAME: pmr0_seed10_iid


In [21]:
!mkdir -p outputs/rounds

run_args = ["--pmr", str(PMR), "--seed", str(SEED), "--partition", PARTITION]
if PARTITION == "dirichlet":
    run_args += ["--dirichlet_alpha", str(DIRICHLET_ALPHA)]

print("Running: cifar10_crowdguard.py", *run_args)
run_checked("cifar10_crowdguard.py", *run_args, log="outputs/v2_full_run.log")

Running: cifar10_crowdguard.py --pmr 0 --seed 10 --partition iid
/kaggle/working/crowdguard-env/lib/python3.10/site-packages/metaflow/metaflow_config.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Aggregator step "start" registered
Collaborator step "train" registered
Aggregator step "collect_models" registered
Collaborator step "local_validation" registered
Aggregator step "collect_commitments" registered
Collaborator step "reveal_votes" registered
Aggregator step "defend" registered
Aggregator step "end" registered

  0%|          | 0.00/170M [00:00<?, ?B/s]
  0%|          | 32.8k/170M [00:00<30:13, 94.0kB/s]
  0%|          | 65.5k/170M [00:00<30:26, 93.3kB/s]
  0%|          | 98.3k/170M [00:01<30:25, 93.4kB/s]
  0%|          | 131k/170M [00:01<30:24, 93.4kB/s] 
 

KeyboardInterrupt: 

## 8b. Confirm the run used the parameters you set

This guards against a version whose parameters were edited underneath it (the duplicate-PMR
mix-up). The notebook stops here if the handoff's own config disagrees with the parameters above.

In [ ]:
import json

expected = {"pmr": PMR, "seed": SEED, "partition": PARTITION}
if PARTITION == "dirichlet":
    expected["dirichlet_alpha"] = DIRICHLET_ALPHA

check_code = f"""
import json, torch
expected = json.loads({json.dumps(json.dumps(expected))})
h = torch.load("outputs/crowdguard_handoff.pt", map_location="cpu", weights_only=True)
c = h["config"]
bad = {{k: (c.get(k), v) for k, v in expected.items() if c.get(k) != v}}
print("handoff config:", {{k: c.get(k) for k in expected}}, "| malicious clients:", len(h["ground_truth_malicious_clients"]))
assert not bad, f"PARAMETER MISMATCH (handoff, notebook): {{bad}}"
print("OK: the handoff matches the notebook parameters")
"""
run_checked("-c", check_code)

## 9. Inspect completed round checkpoints

After the experiment finishes, check that the expected round artifacts exist.

In [ ]:
from pathlib import Path

output_dir = Path("outputs")
round_dir = output_dir / "rounds"

if round_dir.exists():
    checkpoints = sorted(round_dir.glob("round_*.pt"))
else:
    checkpoints = []

print("Number of completed round checkpoints:", len(checkpoints))

for checkpoint in checkpoints:
    size_mb = checkpoint.stat().st_size / (1024 ** 2)
    print(f"{checkpoint}  ({size_mb:.2f} MB)")

For a completed 10-round experiment, you should expect the corresponding
completed round artifacts.

## 10. Inspect the full-run log

Look for:

- round completion
- detected clients
- commitment collection
- commitment digest
- reveal status
- valid/missing/malformed/hash-mismatch reveals
- final-M selection
- final handoff creation

Do not manually change the experimental output based on console logs.

In [ ]:
!grep -nE \
"detected_clients|detected=|commit|reveal|missing_commitment|missing_reveal|malformed_reveal|hash_mismatch|final_M|frequency|tau|handoff" \
outputs/v2_full_run.log | tail -n 300

## 11. Inspect the output directory

In [ ]:
from pathlib import Path

for path in sorted(Path("outputs").rglob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"{path}  ({size_mb:.2f} MB)")

## 12. Validate the final v2 handoff

This must be run on the completed v2 handoff before interpreting the results.

In [ ]:
# validate_handoff.py takes the handoff path as a required argument;
# run_checked stops the notebook if validation reports any error
run_checked("validate_handoff.py", "outputs/crowdguard_handoff.pt")

## 13. Evaluate the final v2 handoff

This extracts/evaluates the final handoff produced by the v2 experiment.

In [ ]:
# evaluate_handoff.py takes the handoff path as a required argument
# and writes outputs/evaluation.json next to it
run_checked("evaluate_handoff.py", "outputs/crowdguard_handoff.pt")

## 14. Record reproducibility metadata

The v2 run should be tied to the exact Git commit used by Kaggle.

In [ ]:
import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone

git_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    text=True
).strip()

git_branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    text=True
).strip()

checkpoint_count = len(
    list(Path("outputs/rounds").glob("round_*.pt"))
)

metadata = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "git_branch": git_branch,
    "git_commit": git_commit,
    "run_name": RUN_NAME,
    "pmr": PMR,
    "seed": SEED,
    "partition": PARTITION,
    "dirichlet_alpha": DIRICHLET_ALPHA if PARTITION == "dirichlet" else None,
    "pin_commit": PIN_COMMIT or None,
    "python_version": subprocess.check_output(
        [python, "--version"],
        text=True
    ).strip(),
    "checkpoint_count": checkpoint_count,
}

Path("outputs/v2_run_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

print(json.dumps(metadata, indent=2))

# 15. Final v2 checklist

Before recording the v2 experiment as complete:

- [ ] Kaggle GPU was enabled
- [ ] Python 3.10.21 was used
- [ ] OpenFL 1.9 was used
- [ ] Metaflow 2.7.15 was used
- [ ] PyTorch CUDA was available
- [ ] all v2 tests passed
- [ ] local smoke test passed
- [ ] local pipeline smoke test passed
- [ ] 20-client experiment completed
- [ ] all 10 FL rounds completed
- [ ] per-round artifacts exist
- [ ] final handoff exists
- [ ] commit-reveal records exist
- [ ] final-M uses the v2 frequency rule
- [ ] `tau = 0.5` is recorded
- [ ] 2,500 KD-reference samples are recorded
- [ ] 7,500 held-out evaluation samples are recorded
- [ ] ASR uses the held-out evaluation set
- [ ] `validate_handoff.py` passes
- [ ] `evaluate_handoff.py` completes
- [ ] exact Git commit is recorded

After the experiment, create:

`RESULTS_V2.md`

Do **not** modify `RESULTS_V1.md`.

The v1 results remain the completed reference run; the new results should be
reported separately as the v2 experiment.